# Phase 0 Bake-off — Nile-Chat-12B

Phase 0 bake-off candidate — see `README.md`'s "Phase 0: model bake-off" section and `claude.md` §6 for the full architectural context.

**Before running anything**: `Runtime > Change runtime type > A100 GPU`, then confirm the GPU is actually attached with the cell below. A previous run of the original `run_qwen2_5.ipynb` failed with `RuntimeError: Failed to infer device type` and `nvidia-smi: command not found` — that was a CPU-only runtime with no GPU attached, not a vLLM or code problem. Don't skip this check.


In [ ]:
!nvidia-smi


If the cell above errors or shows no GPU, stop here and fix the runtime type before continuing — every cell after this one assumes a real GPU is visible.


In [ ]:
!pip install -q -U vllm transformers accelerate


**Fix for the `torchaudio`/`torchvision`/`torch` CUDA-version saga** — three attempts, each correcting the previous one against real evidence:

1. First attempt: uninstall both `torchvision` and `torchaudio` after installing vLLM (a text-only pipeline shouldn't need either). Wrong — `vllm.transformers_utils.processors.*` imports `torchvision` **unconditionally**, registering multimodal processors for every architecture vLLM supports at package-import time, not just the one being served. This produced `ModuleNotFoundError: No module named 'torchvision'` at server startup, before any weights load — proof `torchvision` is a real, unguarded hard requirement.
2. Second attempt: reinstall both packages from the exact CUDA-tagged index `torch` actually used (detected live, never hardcoded — a prior guess of "cu121/cu124" would have been wrong against a real cu130 install). `torchvision` matched cleanly. `torchaudio` didn't — no matching `cu130` build exists on the official index yet, so pip silently resolved an older `cu128` build, reproducing the original CUDA-mismatch `RuntimeError`.
3. **Current fix**: stop trying to satisfy `torchaudio` at all, and stop asserting it must import cleanly. We have **zero confirmed evidence** anything in vLLM's startup path hard-requires `torchaudio` the way it hard-requires `torchvision` — every `torchaudio` failure we've hit has been a *present-but-CUDA-mismatched* `RuntimeError`, never a clean `ModuleNotFoundError` from it being absent. That distinction matters: most optional-modality imports are guarded with `try: import torchaudio except ImportError:`, which cleanly catches an absent package but does **not** catch a `RuntimeError` from a present-but-broken one. Uninstalling `torchaudio` converts an uncatchable crash into a cleanly-catchable optional-dependency miss — strictly better, not a compromise.

**If you're resuming a Colab session where an earlier cell already crashed vLLM or installed a mismatched `torchaudio`**: `Runtime > Restart session` first, then run from the top — a dead `vllm serve` process can still be holding GPU memory, and a stale kernel can have `torchaudio` cached as broken from before this fix ran.


In [ ]:
# Detect whatever torch build vLLM actually installed, reinstall torchvision (a confirmed
# hard requirement) matched to that exact CUDA-tagged index, and deliberately leave torchaudio
# UNINSTALLED — see the note above for why that's the correct fix, not a workaround.
import torch

torch_version = torch.__version__.split("+")[0]   # strip local suffix, e.g. "2.9.0+cu130" -> "2.9.0"
torch_cuda = torch.version.cuda                    # whatever vLLM actually pulled in
cuda_tag = "cu" + torch_cuda.replace(".", "")       # "13.0" -> "cu130"

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

# Verify in a fresh subprocess (not this kernel) so a stale sys.modules cache can't hide a
# real remaining problem or falsely report one that's already fixed. Only torch/torchvision are
# required to import cleanly — torchaudio being ABSENT is the intended, correct end state.
import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")


**Note on the install line**: uses `-U` (latest) rather than `run_qwen2_5.ipynb`'s pinned `transformers==4.47.1`, since Nile-Chat's base architecture wasn't confirmed during research and an older pinned transformers may not recognize it.

**Plan B — only run the cell below if `vllm serve` (next cell) crashes with `ModuleNotFoundError: No module named 'torchaudio'`.** That would mean torchaudio is *also* a real, unguarded hard requirement somewhere in vLLM's startup path, the same way `torchvision` turned out to be — in which case we need *something* importable named `torchaudio`, even though no real matched-CUDA build exists yet. A plain Python-level `sys.modules` monkey-patch in *this* notebook kernel would not help — `vllm serve` runs as a separate OS process with its own independent module cache, so the stub has to be a real file on disk where that subprocess's own import machinery will find it. Only run this if the specific `ModuleNotFoundError: No module named 'torchaudio'` actually appears — don't run it pre-emptively, since it isn't needed if `torchaudio` was never really required.


In [ ]:
# PLAN B — see the markdown note above. Writes a minimal real stub package to site-packages
# so any bare `import torchaudio` succeeds without needing a real (CUDA-build-sensitive)
# install. Safe here specifically because this whole pipeline is text-only — no request against
# Falcon-H1/Nile-Chat/Qwen should ever actually call a real torchaudio function against it.
import sysconfig, os

site_packages = sysconfig.get_paths()["purelib"]
stub_dir = os.path.join(site_packages, "torchaudio")
os.makedirs(stub_dir, exist_ok=True)
with open(os.path.join(stub_dir, "__init__.py"), "w") as f:
    f.write(
        "__version__ = '0.0.0-stub'\n"
        "# Minimal stub — no real cu130 torchaudio build exists yet, and this text-only\n"
        "# pipeline never calls a real torchaudio function. Exists only to satisfy an\n"
        "# unguarded `import torchaudio` statement somewhere in vLLM's startup path.\n"
    )
print(f"Wrote stub torchaudio package to {stub_dir} — re-run the vllm serve cell now.")


In [ ]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name nile-chat-12b \
    > vllm.log 2>&1 &

# No --enforce-eager here: that flag was specifically needed for Falcon-H1's hybrid Mamba2
# architecture (CUDA graph capture SIGKILLed the process — see README's Phase 0 verdict).
# Nile-Chat-12B is confirmed built on Gemma 3 (a standard dense transformer, verified from its
# own real generation_config.json), which has no comparable known vLLM graph-capture gap — so
# this doesn't preemptively trade away throughput for a problem specific to a different model's
# architecture. If vLLM errors with an unrecognized-architecture message anyway, uncomment:
#   --trust-remote-code


In [ ]:
# Poll instead of a fixed sleep — a 34B/12B model can take far longer to load than the
# 7B-AWQ setup's original 90s sleep, and a fixed sleep either wastes time or times out
# too early depending on the model. This polls the actual log for the server coming up.
import time

ready = False
for attempt in range(60):  # up to 10 minutes
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log and "ERROR" in log:
        print("vLLM logged an error while loading — check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"nile-chat-12b","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'


If the curl call above didn't return a real completion, stop and fix it before opening a tunnel — a tunnel just exposes whatever's on :8001, broken or not.


In [ ]:
# Robust cloudflared quick-tunnel setup. The original notebook's tunnel cell failed
# silently (`wget -q` swallowed the real download error, then chmod failed with a
# confusing "cannot access" message) — this version checks each step explicitly and
# polls the log for the URL instead of a single fixed sleep.
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed — re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s — check cloudflared.log for errors and re-run this cell"
print(f"Tunnel URL: {tunnel_url}")
print("Set in your local src/.env:")
print(f"  GENERATION_BASE_URL={tunnel_url}")
print("  GENERATION_MODEL_NAME=nile-chat-12b")


In [ ]:
# Diagnostics — useful if anything above looked wrong.
!ps aux | grep vllm
print("---")
!nvidia-smi


In [ ]:
# Final local smoke test — re-run any time to confirm the server is still alive
# (Colab can reclaim idle GPUs; if this stops responding, re-run the vllm serve cell).
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"nile-chat-12b","messages":[{"role":"user","content":"إزيك؟"}],"max_tokens":32}'
